## Phase 1 — Baseline

### Pre-processing
Audio ricampionato a 16 kHz, convertito in mono e riempito/troncato a una durata fissa di 4 secondi. Ogni forma d'onda viene normalizzata in base al picco prima dell'estrazione delle caratteristiche. Vengono calcolati spettrogrammi log-mel (64 bin mel) e convertiti in scala dB, seguiti dalla standardizzazione globale (media zero, varianza unitaria).

### Network architecture
Il modello di base è una CRNN composta da tre blocchi convoluzionali 2D (Conv–BatchNorm–ReLU–Pooling–Dropout) per l'estrazione delle caratteristiche tempo-frequenza, seguiti da un LSTM bidirezionale per la modellazione temporale. Il pooling della media temporale produce un embedding globale, che viene inviato a un classificatore a due livelli completamente connesso. La forma di input è `[B, 1, 64, T]`, l'output è logit su 8 classi di emozioni.

### Validation protocol and metrics
Viene utilizzata la cross-validazione indipendente dal parlante: i set di training, validazione e test contengono parlanti disgiunti (GroupKFold). All'interno di ogni fold, i parlanti di validazione vengono separati anche dai modelli di training. I modelli vengono selezionati in base alla migliore accuratezza di validazione. La valutazione viene eseguita utilizzando l'accuratezza del test, la precisione/recall/F1 per classe e le matrici di confusione. L'entropia incrociata ponderata per classe con smoothing delle etichette viene adottata per mitigare lo squilibrio di classe.

## Phase 2 — Improvements

### Augmentation
Data augmentation viene applicata solo al training set, includendo guadagno casuale, time shift, rumore additivo (basato su SNR) e SpecAugment (mascheramento temporale e di frequenza su spettrogrammi log-mel). Queste tecniche simulano una variabilità acustica realistica e migliorano la robustezza rispetto alla baseline.

### Speaker-independent evaluation
La suddivisione indipendente dal parlante impone la generalizzazione su parlanti non rilevati, prevenendo la dispersione dei parlanti e fornendo uno scenario di valutazione più realistico.

### Quantitative comparison
I modelli vengono confrontati utilizzando l'accuratezza del test, il punteggio macro-F1 e le matrici di confusione tra le pieghe, evidenziando l'impatto dell'aumento e della valutazione indipendente dal parlante rispetto alla linea di base.

### Qualitative error analysis
L'analisi degli errori si concentra sulle confusioni sistematiche tra emozioni simili (ad esempio, felice vs sorpreso, neutro vs calmo), sulla variabilità tra parlanti e sui potenziali bias del dataset. Le matrici di confusione e le visualizzazioni di incorporamento (t-SNE) vengono utilizzate per esaminare i modelli di clustering e di classificazione errata.


In [ ]:
# IMPORTS
import os
from collections import Counter
import numpy as np
import torch
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, confusion_matrix

from src.models.crnn import CRNN
from src.preprocessing.dataset import (
    list_ravdess_files,
    filter_audio_speech,
    parse_ravdess_filename,
    extract_label_idx,
    RavdessDataset,
    IDX2LABEL,
)
from src.utils import set_seed, evaluate

# Helpers 
from src.experiment_config import ExperimentConfig
from src.splitting import make_cv_splits, split_train_val_within_fold


In [ ]:
# CONFIGURAZIONE
DATA_ROOT = os.path.join(os.getcwd(), "data")

# Hyperparameters, qui i parametri sono già definiti in experiment_config.py ma possono essere sovrascritti
N_FOLDS = 6
BATCH_SIZE = 64
EPOCHS = 70
LR = 1e-3
WEIGHT_DECAY = 1e-4

SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# TOGGLES
cfg = ExperimentConfig(
    speaker_independent=True,   
    augmentation=True,          
    n_folds=N_FOLDS,
    seed=SEED,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

aug_cfg = cfg.build_aug_cfg()

print("CONFIG:")
print("speaker_independent =", cfg.speaker_independent)
print("augmentation        =", cfg.augmentation)
print("n_folds             =", cfg.n_folds)


In [ ]:
# Lista file + labels + actors

all_files = list_ravdess_files(DATA_ROOT)
all_files = filter_audio_speech(all_files)

labels = [extract_label_idx(fp) for fp in all_files]
actors = [parse_ravdess_filename(fp)["actor"] for fp in all_files]

print(f"Tot files: {len(all_files)}")
print(f"Distribuzione classi: {Counter(labels)}")
print(f"Numero attori unici: {len(set(actors))}")


In [ ]:
# Cross-validation (speaker-independent ON/OFF via cfg)
fold_results = []

print(f"\n{'='*60}")
print(f"CROSS-VALIDATION CON {cfg.n_folds} FOLDS")
print(f"Speaker-independent: {cfg.speaker_independent}")
print(f"Augmentation:        {cfg.augmentation}")
print(f"{'='*60}\n")

for fold_idx, train_val_idx, test_idx in make_cv_splits(
    all_files=all_files,
    labels=labels,
    actors=actors,
    n_folds=cfg.n_folds,
    speaker_independent=cfg.speaker_independent,
    seed=cfg.seed,
):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/{cfg.n_folds}")
    print(f"{'='*60}")

    # SPLIT DEI FILE
    test_files = [all_files[i] for i in test_idx]
    train_val_files = [all_files[i] for i in train_val_idx]

    train_val_actors = [actors[i] for i in train_val_idx]
    train_val_labels = [labels[i] for i in train_val_idx]

    test_actors = sorted(set([actors[i] for i in test_idx]))

    # ULTERIORE SPLIT TRAIN/VAL
    split = split_train_val_within_fold(
        train_val_files=train_val_files,
        train_val_labels=train_val_labels,
        train_val_actors=train_val_actors,
        speaker_independent=cfg.speaker_independent,
        fold_idx=fold_idx,
        seed=cfg.seed,
    )

    train_files = split["train_files"]
    val_files = split["val_files"]
    train_actors = split["train_actors"]
    val_actors = split["val_actors"]

    print(f"Train: {len(train_files)} files, {len(train_actors)} speakers {train_actors}")
    print(f"Val:   {len(val_files)} files, {len(val_actors)} speakers {val_actors}")
    print(f"Test:  {len(test_files)} files, {len(test_actors)} speakers {test_actors}")

    # CLASS WEIGHTS
    train_labels_fold = [extract_label_idx(fp) for fp in train_files]
    counts = Counter(train_labels_fold)

    weights = torch.tensor([1.0 / counts[i] for i in range(8)], dtype=torch.float, device=device)
    weights = weights / weights.sum() * 8

    print(f"Class counts: {counts}")
    print(f"Weights: {weights.detach().cpu().numpy().round(3)}")

    # DATASET E DATALOADER
    AUG_ON = cfg.augmentation

    train_ds = RavdessDataset(train_files, augmentation=AUG_ON, aug_config=aug_cfg)
    val_ds   = RavdessDataset(val_files,   augmentation=False)
    test_ds  = RavdessDataset(test_files,  augmentation=False)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                              num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,
                              num_workers=0, pin_memory=True)

    # MODEL, OPTIMIZER, SCHEDULER
    model = CRNN(n_classes=8, n_mels=64).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=6
    )

    best_val_acc = 0.0
    best_path = f"best_fold_{fold_idx}.pt"

    val_acc_hist = []
    test_acc_hist = []
    val_loss_hist = []
    test_loss_hist = []

    print(f"\nInizio training fold {fold_idx + 1}...")

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = torch.nn.functional.cross_entropy(
                logits, y, weight=weights, label_smoothing=0.1
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            running_loss += loss.item() * y.size(0)
            running_correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)

        train_loss = running_loss / total
        train_acc = running_correct / total

        val_loss, val_acc = evaluate(model, val_loader, device)
        test_loss_epoch, test_acc_epoch = evaluate(model, test_loader, device)

        scheduler.step(val_acc)

        val_acc_hist.append(val_acc)
        test_acc_hist.append(test_acc_epoch)
        val_loss_hist.append(val_loss)
        test_loss_hist.append(test_loss_epoch)

        if epoch % 5 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:3d}/{cfg.epochs} | "
                f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
                f"val loss {val_loss:.4f} acc {val_acc:.4f} | "
                f"test acc {test_acc_epoch:.4f}"
            )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

    # VALUTAZIONE FINALE DEL FOLD
    model.load_state_dict(torch.load(best_path, map_location=device))

    final_test_loss, final_test_acc = evaluate(model, test_loader, device)

    print(f"\n{'='*60}")
    print(f"RISULTATI FOLD {fold_idx + 1}")
    print(f"{'='*60}")
    print(f"Best Validation Accuracy: {best_val_acc:.4f}")
    print(f"Final Test Accuracy:      {final_test_acc:.4f}")

    # Classification report
    y_true = []
    y_pred = []
    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            logits = model(x).cpu()
            preds = logits.argmax(dim=1).numpy().tolist()
            y_pred.extend(preds)
            y_true.extend(y.numpy().tolist())

    print("\nClassification Report:")
    target_names = [IDX2LABEL[i] for i in range(8)]
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

    # VISUAL ANALYSIS (PER FOLD) 
    from src.visual_analysis.visual_analysis import (
        extract_crnn_embeddings,
        tsne_project,
        plot_tsne_by_label,
        plot_tsne_errors_and_speakers
    )

    # t-SNE su embeddings del test set (fold corrente)
    embs, ys, preds, paths = extract_crnn_embeddings(model, test_loader, device, return_paths=True)
    Z = tsne_project(embs, pca_dim=50, tsne_perplexity=30, seed=42)

    plot_tsne_by_label(Z, ys, title=f"Fold {fold_idx+1} — t-SNE (true labels)")
    plot_tsne_errors_and_speakers(Z, ys, preds, paths, title=f"Fold {fold_idx+1} — t-SNE (errors + speaker id)")

    # SALVA RISULTATI DEL FOLD
    fold_results.append({
        "fold": fold_idx,
        "test_speakers": test_actors,
        "val_speakers": val_actors,
        "train_speakers": train_actors,
        "best_val_acc": best_val_acc,
        "final_test_acc": final_test_acc,
        "val_acc_history": val_acc_hist,
        "test_acc_history": test_acc_hist,
        "val_loss_history": val_loss_hist,
        "test_loss_history": test_loss_hist,
        "y_true": y_true,
        "y_pred": y_pred,
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    })


## SUMMARY + PLOTS

In [ ]:

test_accs = [fr["final_test_acc"] for fr in fold_results]
val_accs  = [fr["best_val_acc"] for fr in fold_results]

print("\n==== SUMMARY ====")
print("speaker_independent:", cfg.speaker_independent)
print("augmentation:       ", cfg.augmentation)
print(f"Val acc (best)  mean={np.mean(val_accs):.4f} std={np.std(val_accs):.4f}")
print(f"Test acc        mean={np.mean(test_accs):.4f} std={np.std(test_accs):.4f}")


# Plot accuracy curves per fold
import matplotlib.pyplot as plt
import numpy as np

for fr in fold_results:
    fold = fr["fold"]
    val_acc = fr["val_acc_history"]
    test_acc = fr["test_acc_history"]
    epochs = np.arange(1, len(val_acc) + 1)

    plt.figure()
    plt.plot(epochs, val_acc, label="val acc")
    plt.plot(epochs, test_acc, label="test acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(f"Fold {fold}: Val/Test accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()


# Plot loss curves per fold
for fr in fold_results:
    fold = fr["fold"]
    val_loss = fr["val_loss_history"]
    test_loss = fr["test_loss_history"]
    epochs = np.arange(1, len(val_loss) + 1)

    plt.figure()
    plt.plot(epochs, val_loss, label="val loss")
    plt.plot(epochs, test_loss, label="test loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"Fold {fold}: Val/Test loss")
    plt.legend()
    plt.tight_layout()
    plt.show()